# L14 – Lösningar

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import keras
from keras import layers

(x_full, y_full), _ = keras.datasets.fashion_mnist.load_data()
perm = np.random.default_rng(42).permutation(len(x_full))
x_train, y_train = x_full[perm[:50_000]], y_full[perm[:50_000]]
x_val, y_val = x_full[perm[50_000:]], y_full[perm[50_000:]]
MEAN, STD = (x_train / 255.0).mean(), (x_train / 255.0).std()
normalize = lambda x: ((x / 255.0 - MEAN) / STD).astype("float32")
x_train, x_val = normalize(x_train), normalize(x_val)

def make_callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)]

def compile_model(model, lr=1e-3):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])
    return model

### Övning 1

In [ ]:
keras.utils.set_random_seed(42)
reg_model = compile_model(keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(512), layers.BatchNormalization(), layers.Activation("relu"), layers.Dropout(0.3),
    layers.Dense(256), layers.BatchNormalization(), layers.Activation("relu"), layers.Dropout(0.3),
    layers.Dense(10),
]))
reg_history = reg_model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=30,
                            batch_size=128, callbacks=make_callbacks(), verbose=2)
assert max(reg_history.history["val_accuracy"]) >= 0.89
print("Rätt!", round(max(reg_history.history["val_accuracy"]), 3))

Ordningen Dense, BatchNormalization, ReLU och Dropout är samma som i L07. `Dense` får ingen aktivering direkt, så att normaliseringen sker före ReLU.

### Övning 2

In [ ]:
keras.utils.set_random_seed(42)
cnn = compile_model(keras.Sequential([
    keras.Input(shape=(28, 28, 1)),                                  # kanalerna SIST
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),                                          # 28 → 14
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),                                          # 14 → 7
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(10),
]))
cnn_history = cnn.fit(x_train[..., None], y_train, validation_data=(x_val[..., None], y_val),
                      epochs=15, batch_size=128, callbacks=make_callbacks(), verbose=2)
assert max(cnn_history.history["val_accuracy"]) >= 0.90
print("Rätt!", round(max(cnn_history.history["val_accuracy"]), 3))

### Övning 3 – förslag på svar
1. `validation_data` motsvarar `val_loader` som skickas till `fit`. `EarlyStopping(patience=..., restore_best_weights=True)` motsvarar `patience=` i `dlkit.fit`, som också återställer de bästa vikterna.
2. Keras tror att utdata är sannolikheter och tillämpar ingen softmax. Logits tolkas fel, och lossen blir felaktig eller numeriskt instabil. Modellen tränar sämre utan felmeddelande.
3. För en **vanlig bildklassificerare i en app** fungerar båda. Keras ger kort kod och enkel export till mobil (TF Lite), och PyTorch har ExecuTorch. Välj det teamet och produktionsmiljön redan använder. För **forskning och nya idéer** väljer de flesta PyTorch: den egna loopen ger full kontroll, felsökningen är enkel och de flesta nya artiklar publicerar PyTorch-kod.